# VietMedBridge — 01f: Trích bài Long Châu từ HTML đã lưu

Dùng **runtime CPU mới** và cùng Drive data root. Notebook đọc experiment
01e đã chạy đủ 11 URL, kiểm hashes, rồi trích lại các HTML đã có. Giữ
heading, đoạn dẫn, paragraph, list, table và chú thích trong bài; loại
cookie, menu, tiểu sử và bài liên quan nằm ngoài vùng nội dung.

Có checkpoint theo doc_id và gói review riêng. RAW, text cũ và corpus
Stage A được giữ để so sánh. Chưa tự nhập/gộp vào canonical corpus.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "recovery_extract_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Đọc experiment đủ 11 URL và trích lại từ cache


In [ ]:
import csv
from vietmedbridge.probe_review import review_browser_captures
from vietmedbridge.artifacts import read_json

SOURCE_KEY = "4c49d3358aa818b7"
EXPECTED_ATTEMPTS_SHA256 = "c4c716d838f4c676e62cc7e9e8e9b8ba366ee04dd5f3e5d2abdc3560b3fdb807"
REPORT_DIR = DATA_ROOT / "reports/crawl_recovery/stage-a-v2"
PROBE_DIR = REPORT_DIR / "experiments" / SOURCE_KEY
with (CHECKOUT / "reports/stage-a-v2-longchau-access-gap.csv").open(encoding="utf-8") as stream:
    OFFICIAL_PAIRS = {int(r["doc_id"]): r["url"] for r in csv.DictReader(stream)}
REVIEW = review_browser_captures(
    PROBE_DIR, REPORT_DIR / "extraction_review",
    expected_links=OFFICIAL_PAIRS,
    expected_attempts_sha256=EXPECTED_ATTEMPTS_SHA256,
    work_dir=WORK_DIR,
)
REVIEW_DIR = Path(REVIEW["output_dir"])
print(json.dumps({k: REVIEW[k] for k in [
    "input_ids", "ready_for_review", "states", "output_dir",
]}, ensure_ascii=False, indent=2))


## 2. Xem text đã làm sạch và tải gói review


In [ ]:
import shutil
import pandas as pd
from IPython.display import display, HTML
from google.colab import files
from vietmedbridge.artifacts import publish_file

display(pd.DataFrame(REVIEW["records"]).reindex(columns=[
    "doc_id", "state", "capture_kind", "source_chars", "section_count",
    "paragraph_count", "quality_flags",
]))
display(HTML((REVIEW_DIR / "review.html").read_text(encoding="utf-8")))
local_zip = Path(shutil.make_archive(
    str(WORK_DIR / f"recovery-extraction-{REVIEW['signature'][:16]}"),
    "zip", root_dir=REVIEW_DIR,
))
target = REPORT_DIR / local_zip.name
publish_file(local_zip, target)
print("Gói text/structure/hash để review:", target)
files.download(str(local_zip))
